# 5. Custom analysis with the Python API

> **Install for this tutorial:** `pip install "scope-profiler[pproc]"`;
> pandas is used for event analysis.

Notebook 2 covered the *aggregates*: how long each region took in total, on
average, per rank. This notebook goes one level down, to the **individual
calls**, which is where you go when the question you have is not one the
built-in summaries answer.

Three tools do most of the work:

| Method | Gives you |
| — | — |
| `results.events()` | one entry per recorded call |
| `results.to_events_dataframe()` | the same, as a pandas DataFrame |
| `results.call_stack()` | the calls with their nesting reconstructed |

As everywhere in this API, times are in **seconds**.

In [ ]:
import os
import tempfile

# Work in a scratch directory so the tutorial leaves no files behind.
os.chdir(tempfile.mkdtemp(prefix="scope-profiler-tutorial-"))

In [ ]:
import time

from scope_profiler import ProfileManager

ProfileManager.setup(file_path="profiling_data.h5")


@ProfileManager.profile("assemble")
def assemble():
    time.sleep(0.002)


with ProfileManager.region("setup"):
    time.sleep(0.01)

for step in range(6):
    with ProfileManager.region("timestep"):
        assemble()
        with ProfileManager.region("solve"):
            # The solve gets slower as the run progresses.
            time.sleep(0.002 + 0.001 * step)
    if step % 3 == 2:
        with ProfileManager.region("checkpoint"):
            time.sleep(0.004)

ProfileManager.finalize(verbose=False)

## Reading the results back in the same script

`ProfileManager.read_results()` opens the file the current configuration just
wrote, so you do not have to repeat the path. Under MPI only rank 0 writes the
merged file, so guard the call with a rank check there.

In [ ]:
results = ProfileManager.read_results()
print(results)
print("same file:", results.file_path.name == "profiling_data.h5")

## One entry per call

`events()` returns the long-form ("tidy") view of the run: every recorded call,
in region order, then rank, then call order.

In [ ]:
events = results.events()

print(f"{len(events)} calls recorded\n")
for event in events[:4]:
    print(event)

Each entry carries the region `name`, the `rank` that recorded it, its
`call_index` within that rank, and `start` / `end` / `duration` in seconds. At `finalize()` the profiler also
assigns every call a `call_id` and the `parent_id` of the call it ran inside
(`-1` for a top level call), so a saved profile carries its call graph explicitly.

### The timeline starts at zero

By default the timestamps are measured from the **start of the run** —
`ProfileManager.setup()` records that instant, and `ProfilingResults` exposes it as
`run_start_time`. The raw values come from `perf_counter_ns()`, a monotonic
clock with an arbitrary origin that is comparable only within one run, and you
can still get them with `relative=False`.

In [ ]:
rel = results.events(include="^timestep$")
raw = results.events(include="^timestep$", relative=False)

print("relative start of the first call [s]:", round(rel[0]["start"], 4))
print(
    "raw start minus run start         [s]:",
    round(raw[0]["start"] - results.run_start_time, 4),
)
print("raw start is a perf_counter value, so it is only comparable within one run")

print("\nwall clock   :", round(results.time_span, 3), "s")

### The time before the first region

`setup()` runs before any region is entered, so the gap between the two is work
the instrumentation never saw — imports, reading input, building a mesh. It is
often the first surprise a profile delivers. The run's origin is the moment
`setup()` is called, so calling it as early as possible is what makes that gap
visible.

In [ ]:
print(
    f"{results.startup_time * 1e3:.2f} ms elapsed before the first region was entered"
)

### Choosing a different origin

`time_origin` is the zero point in use: the registered start time, or the first
region entry for files that have none. Override it per call with `origin`.

The `plot_*` functions are the one exception — they frame their x axis on the
first region entry, so a long startup does not fill a chart with empty space.
The last line below reproduces exactly what a chart's axis shows.

None of this is required reading for older files: one written before `setup()`
recorded a start time reports `run_start_time` as `None`, falls back to the
first region entry, and behaves exactly as it always did.

In [ ]:
print("origin is the run start :", results.time_origin == results.run_start_time)

print(
    "\nfrom run start   :",
    [round(e["start"], 4) for e in results.events(include="solve")],
)
print(
    "from first entry :",
    [
        round(e["start"], 4)
        for e in results.events(include="solve", origin=results.minimum_start_time)
    ],
)

`time_span` is the width of the profiled window, which is what you want when
asking *what fraction of the run went into this region* — the sum of all region
durations double-counts nested regions and can exceed the wall clock.

In [ ]:
for region in results.get_regions():
    share = region.total_duration / results.time_span
    print(f"  {region.name:<11} {share:6.1%} of the run")

### Selecting what you get

`events()` takes the same `include` / `exclude` regexes as the rest of the API,
plus `ranks`:

```python
results.events(include="solve")             # one region
results.events(include=["solve", "assemble"])
results.events(exclude="checkpoint")
results.events(ranks=0)                     # one rank, or ranks=[0, 2]
```

In [ ]:
solve_events = results.events(include="solve")
print("calls  :", len(solve_events))
print("indices:", [e["call_index"] for e in solve_events])
print("slowest:", max(solve_events, key=lambda e: e["duration"]))

## As a DataFrame

`to_events_dataframe()` returns the same data with one row per call, which is
usually the shortest path from a question to an answer. It needs pandas (part
of the `pproc` extra).

In [ ]:
frame = results.to_events_dataframe()
frame.head()

From here every question is a groupby. A few that come up constantly:

In [ ]:
# How consistent is each region, call to call?
frame.groupby("name")["duration"].agg(["count", "mean", "std", "max"]).sort_values(
    "mean", ascending=False
)

In [ ]:
# The ten slowest individual calls in the whole run.
frame.nlargest(10, "duration")[["name", "rank", "call_index", "start", "duration"]]

In [ ]:
# Time spent per region, per rank - the shape load-imbalance analysis wants.
frame.pivot_table(index="rank", columns="name", values="duration", aggfunc="sum")

## Down to one region

`Region` and `MPIRegion` expose the same view for a single region, so you can
narrow before you widen. `Region` also hands back the stored integers directly,
for the rare case where the nanosecond values matter more than the convenience
of seconds.

In [ ]:
solve = results["solve"]  # an MPIRegion: the region across all ranks
print("all ranks   :", len(solve.events()), "calls")
print("rank 0 only :", len(solve.events(ranks=0)), "calls")

rank0 = solve[0]  # a Region: one rank
print("\nseconds     :", rank0.durations[:3])
print("nanoseconds :", rank0.durations_ns[:3])
print("start (ns)  :", rank0.start_times_ns[:3] - rank0.start_times_ns[0])

## The call stack

Every call is stored as a `(start, end)` pair, so `call_stack()` **reconstructs**
the nesting from containment: a call that starts while another is still open is
treated as its child. That is what the flame graph draws, and `call_stack()`
hands you the same structure as plain dicts — each with its `depth`, its inclusive
and exclusive duration, and the `call_path` of region names leading to it.

In [ ]:
calls = results.call_stack(rank=0)

for call in calls[:8]:
    indent = "  " * call["depth"]
    print(f"{indent}{call['name']:<12} {call['duration'] * 1e3:6.2f} ms")

Every call carries its `depth` and the **index** of its `parent` in the returned
list. Indices rather than names, because a region called repeatedly — or
recursively — contributes several entries under one name.

`call_stack_roots()` and `call_stack_children()` turn that flat list into a tree
you can walk:

In [ ]:
from scope_profiler import call_stack_children, call_stack_roots

children = call_stack_children(calls)

for root in call_stack_roots(calls):
    names = [calls[child]["name"] for child in children[root]]
    print(f"{calls[root]['name']:<12} contains {names}")

### Self time

A useful thing the aggregates cannot give you per call: **exclusive** (self)
time — how long a region took on its own, with the time spent inside its children
removed. Every call carries it as `exclusive_duration`, next to
`inclusive_duration`; it is what the tree above gives you by subtracting the
children.

In [ ]:
from collections import defaultdict

total = defaultdict(float)
self_time = defaultdict(float)

for call in calls:
    total[call["name"]] += call["inclusive_duration"]
    self_time[call["name"]] += call["exclusive_duration"]

print(f"{'region':<12}{'total [ms]':>12}{'self [ms]':>12}")
for name in sorted(total, key=total.get, reverse=True):
    print(f"{name:<12}{total[name] * 1e3:12.2f}{self_time[name] * 1e3:12.2f}")

`timestep` is nearly all children — it is a container. The regions with large
self time are the ones actually doing work, and the ones worth optimising.

## Several ranks

Everything above is rank-aware; a serial run just happens to have one rank. To
show the shape without launching `mpirun`, here is a two-rank file written by
hand — it also documents the on-disk layout, which is plain HDF5:

In [ ]:
import numpy as np

from scope_profiler import MPIRegion, ProfilingResults, Region

NS = 1_000_000_000  # timestamps are stored as int64 nanoseconds
starts = (np.array([0.0, 0.5, 1.0]) * NS).astype(np.int64)
ranks = {}
for rank, factor in enumerate([1.0, 1.6]):  # rank 1 is the slow one
    durations = (np.array([0.2, 0.25, 0.2]) * factor * NS).astype(np.int64)
    ranks[rank] = Region(starts, starts + durations)

mpi_reader = ProfilingResults({"solve": MPIRegion("solve", ranks)}, num_ranks=2)
print(mpi_reader)

mpi_reader.to_events_dataframe()

In [ ]:
# Load imbalance: how much longer did the slowest rank take than the fastest?
per_rank = mpi_reader["solve"].total_durations()
print("total per rank:", per_rank)
print(f"imbalance     : {max(per_rank.values()) / min(per_rank.values()):.2f}x")

## Next

[6. Building your own plots](/scope-profiler/guide/06-custom-plots/) takes the same three tools
and draws charts with them.